# linear_panel_fd

> 对应代码：`EconometricsCode/code_in_notes/Chap.14/linear_panel_fd.do`（本 notebook 中的 Stata 代码与 do 文件一致）
>
> 运行方式：通过 *PyStata* 在 Jupyter 中调用 Stata，安装与使用说明见 `notebooks/PyStata.md`。

**说明**：为在 notebook 中直接内联显示图形，本 notebook 的 Stata 代码省略了 `.do` 文件中的 `graph export` 导出命令（`code_in_notes` 中对应的 .do 文件保留导出，用于生成书中的插图）。

首先启动 Stata 环境，并把工作目录切换到 `code_in_notes/Chap.14`，这样 Stata 中的相对路径（如 `../datasets/...`）与同名 do 文件完全一致，图片也会输出到该目录。

In [ ]:
# 配置 PyStata：启动 Stata 并注册 %%stata 魔法命令
import os
os.chdir("../../code_in_notes/Chap.14")

import stata_setup
stata_setup.config("/opt/Stata", "mp")

我们先清空内存并关闭分页显示，然后读入县级选举与报纸数据 voting_cnty_clean.dta。

In [ ]:
%%stata
clear
set more off
use ../datasets/voting_cnty_clean.dta

用 xtset 声明面板结构：截面单位为县（cnty90），时间维度为 year，声明之后才能使用 L.、D. 等时间序列算子。

In [ ]:
%%stata
xtset cnty90 year

生成解释变量与被解释变量的 4 期滞后（L4. 表示滞后 4 期，对应数据结构中的两届选举间隔），再用当期值减去滞后值得到一阶差分变量 changedailies 与 changeprestout。一阶差分的作用是消掉不随时间变化的县层面不可观测因素（如长期的政治文化、地理禀赋），只要这些遗漏因素不随时间变化，差分后它们就被完全消除。

In [ ]:
%%stata
gen numdailies_l1=L4.numdailies
gen prestout_l1=L4.prestout
gen changedailies=numdailies-numdailies_l1
gen changeprestout=prestout-prestout_l1

在差分后的方程上做回归：被解释变量是投票率的变化量，核心解释变量是日报数量的变化量。absorb(i.st#i.year) 吸收了州×年份的联合固定效应，控制每个州每一年的共同冲击；cluster(cnty90) 把标准误在县层面聚类，允许同一县内误差项任意相关；resid 保存残差。if mainsample 限定在主样本内估计。

In [ ]:
%%stata
// 一阶差分
reghdfe changeprestout changedailies if mainsample, absorb(i.st#i.year) cluster(cnty90) resid